# 09 — Kiểm chứng cách HCO và VNLS chọn đặc trưng

**Ba câu hỏi:** Mask thay đổi thế nào khi đổi seed? Có hơn mask ngẫu nhiên cùng số cột không? Khi bỏ một cột đã chọn, AP trên train-CV thay đổi thế nào? Đây là nghiên cứu khám phá; dữ liệu validation và test từng được dùng trong dự án, không phải một tập mới độc lập.

Thực nghiệm khóa trước khi chạy: cùng LightGBM 80 cây của notebook 05b; cutoff 50% và 75%; train chia 3 GroupKFold theo sinh viên; năm seed 41, 77, 103, 211, 307; mỗi HCO, VNLS, HCO–VNLS được chấm tối đa 8 mask khác nhau; alpha = 0,85. Mỗi mask cuối đối chiếu với 8 mask ngẫu nhiên **cùng số cột**; không báo cáo riêng mask ngẫu nhiên tốt nhất. Chỉ số trên validation chỉ mô tả; tuyệt đối không lấy seed tốt nhất để kết luận như test.

In [ ]:
from pathlib import Path
from itertools import combinations
import sys
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "research_code").exists() and (candidate / "shared_code").is_dir()), Path.cwd())
assert (ROOT / "research_code").is_dir(), "Open Jupyter from packages/ml or its notebooks directory."
sys.path.insert(0, str(ROOT))
from shared_code.thuc_nghiem import nap_du_lieu, chon_dac_trung, KET_QUA, HINH
from research_code.models import ModelConfig, make_pipeline
from research_code.evaluation import metric_table
from research_code.optimizers import MaskObjective, run_search_budgeted
SEEDS = (41, 77, 103, 211, 307)
CUTOFFS = (.50, .75)
METHODS = ('HCO', 'VNLS', 'HCO-VNLS')
BUDGET, RANDOM_PER_MASK, ALPHA = 8, 8, .85
CONFIG = ModelConfig('LightGBM_search_80', LGBMClassifier(n_estimators=80, learning_rate=.055,
    num_leaves=15, min_child_samples=35, verbosity=-1, random_state=41, n_jobs=4))
KET_QUA.mkdir(parents=True, exist_ok=True)
HINH.mkdir(parents=True, exist_ok=True)
print('Giao thức: 2 cutoff × 5 seed × 3 phương pháp × budget 8; 8 mask ngẫu nhiên/cấu hình.')

## 1. Một mask được chấm như thế nào?

Mỗi mask là 38 bit: 1 giữ cột, 0 bỏ cột. Tiền xử lý và LightGBM được fit riêng trong từng fold train; AP CV là trung bình AP của ba fold **chỉ trong train**. Thuật toán tối đa hóa `fitness = 0,85 × AP_CV + 0,15 × (1 − số_cột/38)`. Đó không phải AP validation. HCO của *dự án* cập nhật vị trí/vận tốc rồi lấy mẫu bit theo sigmoid; VNLS lật một bit quanh mask tốt nhất; hybrid HCO–VNLS thêm bước tìm kiếm lân cận. Không tuyên bố đây là bản tái hiện chính xác một bài báo HCO cụ thể.

Nhật ký ghi **mỗi mask ứng viên mới** cùng AP CV, fitness, số cột và các tên cột. Mức `best_fitness_cv` lặp lại khi một ứng viên mới bị từ chối. Ở HCO, hai ứng viên liền kề có thể thuộc hai cá thể khác nhau, không phải một phép lật bit trực tiếp.

In [ ]:
class TracedObjective(MaskObjective):
    def __init__(self, frame, labels, groups, config, alpha):
        super().__init__(frame, labels, groups, config, alpha)
        self.records = []

    def __call__(self, mask):
        old_calls = self.calls
        fitness_cv, ap_cv = super().__call__(mask)
        if self.calls > old_calls:
            selected = self.frame.columns[np.flatnonzero(mask)].tolist()
            self.records.append(dict(call=self.calls, features=len(selected), ap_cv=ap_cv,
                fitness_cv=fitness_cv, selected_columns='|'.join(selected)))
        return fitness_cv, ap_cv


def validation_metrics(train_features, train_labels, valid_features, valid_labels, selected):
    model = make_pipeline(train_features[selected], CONFIG)
    model.fit(train_features[selected], train_labels)
    return metric_table(valid_labels, model.predict_proba(valid_features[selected])[:, 1])

In [ ]:
started = time.perf_counter()
search_rows, random_rows, trace_rows = [], [], []
for cutoff in CUTOFFS:
    frame = nap_du_lieu(cutoff)
    features = chon_dac_trung(frame)
    train = frame.loc[frame.split == 'train']
    valid = frame.loc[frame.split == 'validation']
    assert len(features) == 38 and set(train.id_student).isdisjoint(valid.id_student)
    train_features, valid_features = train[features], valid[features]
    train_labels, valid_labels = train.risk.to_numpy(), valid.risk.to_numpy()
    groups = train.id_student.to_numpy()
    full_objective = MaskObjective(train_features, train_labels, groups, CONFIG, ALPHA)
    full_fitness, full_ap_cv = full_objective(np.ones(len(features), dtype=np.int8))
    search_rows.append(dict(cutoff=cutoff, method='Đủ 38 cột', seed=np.nan, features=38,
        ap_cv=full_ap_cv, fitness_cv=full_fitness, calls=0, seconds=np.nan,
        selected_columns='|'.join(features),
        **validation_metrics(train_features, train_labels, valid_features, valid_labels, features)))
    for seed in SEEDS:
        for method_index, method in enumerate(METHODS):
            objective = TracedObjective(train_features, train_labels, groups, CONFIG, ALPHA)
            search = run_search_budgeted(method, objective, seed=seed, budget=BUDGET, population=3)
            selected = search['selected_columns']
            assert search['objective_calls'] <= BUDGET and len(selected) == search['selected_count']
            search_rows.append(dict(cutoff=cutoff, method=method, seed=seed, features=len(selected),
                ap_cv=search['average_precision_cv'], fitness_cv=search['fitness_cv'],
                calls=search['objective_calls'], seconds=search['wall_seconds'],
                selected_columns='|'.join(selected),
                **validation_metrics(train_features, train_labels, valid_features, valid_labels, selected)))
            trace_rows.extend(dict(cutoff=cutoff, method=method, seed=seed, **record)
                              for record in objective.records)
            generator = np.random.default_rng(20260926 + int(cutoff * 100) * 10000 + seed * 10 + method_index)
            random_objective = MaskObjective(train_features, train_labels, groups, CONFIG, ALPHA)
            for random_index in range(RANDOM_PER_MASK):
                random_selected = sorted(generator.choice(features, len(selected), replace=False).tolist())
                mask = np.isin(features, random_selected).astype(np.int8)
                random_fitness, random_ap_cv = random_objective(mask)
                random_rows.append(dict(cutoff=cutoff, method=method, seed=seed,
                    random_index=random_index, features=len(selected), ap_cv=random_ap_cv,
                    fitness_cv=random_fitness, selected_columns='|'.join(random_selected),
                    **validation_metrics(train_features, train_labels, valid_features, valid_labels, random_selected)))
        print(f'{cutoff:.0%} — seed {seed}: xong ba phương pháp và mask ngẫu nhiên cùng kích thước.')
search_frame, random_frame, trace_frame = map(pd.DataFrame, (search_rows, random_rows, trace_rows))
trace_frame['best_fitness_cv'] = trace_frame.groupby(['cutoff', 'method', 'seed']).fitness_cv.cummax()
search_frame.to_csv(KET_QUA / 'kiem_chung_mask_validation.csv', index=False, encoding='utf-8-sig')
random_frame.to_csv(KET_QUA / 'mask_ngau_nhien_cung_kich_thuoc.csv', index=False, encoding='utf-8-sig')
trace_frame.to_csv(KET_QUA / 'nhat_ky_mask_hco_vnls.csv', index=False, encoding='utf-8-sig')
print('Đã chấm', len(search_frame) - 2, 'mask thuật toán và', len(random_frame),
      'mask ngẫu nhiên; số phút:', round((time.perf_counter() - started) / 60, 2))

## 2. Nhìn đường đi, độ ổn định và đối chứng

Biểu đồ đầu vẽ fitness tốt nhất theo **số mask mới đã đánh giá** chứ không vẽ AP validation. Tần suất chọn feature và Jaccard giữa các mask của năm seed đo độ ổn định, **không** chứng minh tầm quan trọng nhân quả. `share_random_below_cv` là tỷ lệ trong tám mask ngẫu nhiên cùng số cột có AP CV thấp hơn mask thuật toán; không phải p-value. Vì cả năm seed dùng cùng train và validation, `std_ap_validation` là độ dao động giữa seed, không phải khoảng tin cậy độc lập.

In [ ]:
example = trace_frame.loc[(trace_frame.cutoff == .75) & (trace_frame.seed == 41)]
display(example[['method', 'call', 'features', 'ap_cv', 'fitness_cv', 'best_fitness_cv']].round(4))
for method in METHODS:
    mask = search_frame.loc[(search_frame.cutoff == .75) & (search_frame.seed == 41)
        & (search_frame.method == method), 'selected_columns'].iloc[0]
    print(method, 'mask cuối:', mask)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, cutoff in zip(axes, CUTOFFS):
    for method in METHODS:
        series = trace_frame.loc[(trace_frame.cutoff == cutoff) & (trace_frame.seed == 41)
            & (trace_frame.method == method)]
        axis.plot(series['call'], series.best_fitness_cv, marker='o', label=method)
    axis.set(title=f'Cutoff {cutoff:.0%}, seed 41', xlabel='Số mask mới đã chấm trên train-CV',
        ylabel='Fitness tốt nhất đến lúc này', xticks=range(1, BUDGET + 1))
    axis.grid(alpha=.25)
axes[0].legend()
fig.tight_layout()
fig.savefig(HINH / 'kiem_chung_duong_di_mask.png', dpi=160)
plt.show()
comparison_rows, frequency_rows = [], []
for (cutoff, method), group in search_frame.loc[search_frame.method != 'Đủ 38 cột'].groupby(['cutoff', 'method']):
    selected_sets = [set(mask.split('|')) for mask in group.selected_columns]
    jaccard = np.mean([len(left & right) / len(left | right) for left, right in combinations(selected_sets, 2)])
    for feature in search_frame.loc[search_frame.cutoff == cutoff, 'selected_columns'].iloc[0].split('|'):
        frequency_rows.append(dict(cutoff=cutoff, method=method, feature=feature,
            selected_in_five_seeds=sum(feature in selected for selected in selected_sets)))
    for row in group.itertuples(index=False):
        controls = random_frame.loc[(random_frame.cutoff == cutoff) & (random_frame.method == method)
            & (random_frame.seed == row.seed)]
        comparison_rows.append(dict(cutoff=cutoff, method=method, seed=int(row.seed), features=row.features,
            ap_cv=row.ap_cv, ap_validation=row.average_precision_risk, recall_risk=row.recall_risk,
            random_mean_ap_cv=controls.ap_cv.mean(), random_mean_ap_validation=controls.average_precision_risk.mean(),
            share_random_below_cv=(controls.ap_cv < row.ap_cv).mean(),
            share_random_below_validation=(controls.average_precision_risk < row.average_precision_risk).mean(),
            mean_jaccard_five_seeds=jaccard))
comparison, frequency = pd.DataFrame(comparison_rows), pd.DataFrame(frequency_rows)
comparison.to_csv(KET_QUA / 'doi_chung_va_on_dinh_mask.csv', index=False, encoding='utf-8-sig')
frequency.to_csv(KET_QUA / 'tan_suat_chon_feature.csv', index=False, encoding='utf-8-sig')
summary = comparison.groupby(['cutoff', 'method']).agg(mean_features=('features', 'mean'),
    mean_ap_cv=('ap_cv', 'mean'), std_ap_cv=('ap_cv', 'std'),
    mean_ap_validation=('ap_validation', 'mean'), std_ap_validation=('ap_validation', 'std'),
    mean_recall=('recall_risk', 'mean'), random_mean_ap_cv=('random_mean_ap_cv', 'mean'),
    share_random_below_cv=('share_random_below_cv', 'mean'), mean_jaccard=('mean_jaccard_five_seeds', 'mean')).reset_index()
display(summary.round(4))
random_best = random_frame.groupby(['cutoff', 'method', 'seed']).ap_cv.max().rename('best_random_ap_cv').reset_index()
matched = comparison.merge(random_best, on=['cutoff', 'method', 'seed'], validate='one_to_one')
matched['beats_best_of_eight_random_cv'] = matched.ap_cv > matched.best_random_ap_cv
display(matched.groupby(['cutoff', 'method']).beats_best_of_eight_random_cv.sum().rename('số seed thắng mask tốt nhất trong 8 random').to_frame())
random_jaccard_rows = []
for (cutoff, method, random_index), sample in random_frame.groupby(['cutoff', 'method', 'random_index']):
    random_sets = [set(mask.split('|')) for mask in sample.selected_columns]
    similarity = np.mean([len(left & right) / len(left | right) for left, right in combinations(random_sets, 2)])
    random_jaccard_rows.append(dict(cutoff=cutoff, method=method, random_index=random_index,
                                    random_jaccard=similarity))
random_jaccard = pd.DataFrame(random_jaccard_rows)
jaccard_comparison = summary[['cutoff', 'method', 'mean_jaccard']].merge(
    random_jaccard.groupby(['cutoff', 'method']).random_jaccard.agg(['mean', 'std']).reset_index(),
    on=['cutoff', 'method'], validate='one_to_one')
display(jaccard_comparison.round(4))
display(search_frame.loc[search_frame.method == 'Đủ 38 cột',
    ['cutoff', 'features', 'ap_cv', 'average_precision_risk', 'accuracy',
     'precision_risk', 'recall_risk', 'f1_risk']].round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, cutoff in zip(axes, CUTOFFS):
    subset = frequency.loc[(frequency.cutoff == cutoff) & (frequency.method == 'HCO-VNLS')]
    subset = subset.sort_values(['selected_in_five_seeds', 'feature'], ascending=[False, True]).head(12)
    axis.barh(subset.feature.iloc[::-1], subset.selected_in_five_seeds.iloc[::-1])
    axis.set(xlim=(0, 5), xlabel='Số seed chọn (trong 5)', title=f'Hybrid — cutoff {cutoff:.0%}')
fig.tight_layout()
fig.savefig(HINH / 'on_dinh_feature_hybrid.png', dpi=160)
plt.show()

## 3. Bỏ một feature trong mask hybrid 75%

Dùng mask seed 41 đã cố định; bỏ lần lượt từng cột và đo `delta_AP_CV = AP_CV(mask gốc) − AP_CV(mask bỏ cột)`. Delta dương nghĩa là bỏ cột làm giảm AP CV **trong mask ấy**; delta âm là dấu hiệu cột không cần thiết trong tổ hợp ấy. Các cột tương quan có thể thay thế nhau; không diễn giải đây là nhân quả hay tầm quan trọng bất biến.

In [ ]:
frame = nap_du_lieu(.75)
features = chon_dac_trung(frame)
train = frame.loc[frame.split == 'train']
chosen = search_frame.loc[(search_frame.cutoff == .75) & (search_frame.seed == 41)
    & (search_frame.method == 'HCO-VNLS'), 'selected_columns'].iloc[0].split('|')
objective = MaskObjective(train[features], train.risk.to_numpy(), train.id_student.to_numpy(), CONFIG, ALPHA)
base_mask = np.isin(features, chosen).astype(np.int8)
base_ap_cv = objective(base_mask)[1]
ablation_rows = []
for feature in chosen:
    smaller_mask = base_mask.copy()
    smaller_mask[features.index(feature)] = 0
    reduced_ap_cv = objective(smaller_mask)[1]
    ablation_rows.append(dict(feature=feature, ap_cv_original=base_ap_cv,
        ap_cv_without_feature=reduced_ap_cv, delta_ap_cv=base_ap_cv - reduced_ap_cv))
ablation = pd.DataFrame(ablation_rows).sort_values('delta_ap_cv', ascending=False)
ablation.to_csv(KET_QUA / 'ablation_hybrid_75_train_cv.csv', index=False, encoding='utf-8-sig')
display(ablation.round(4))
fig, axis = plt.subplots(figsize=(9, max(4, len(ablation) * .38)))
axis.barh(ablation.feature.iloc[::-1], ablation.delta_ap_cv.iloc[::-1],
    color=['#287b9d' if change > 0 else '#a55244' for change in ablation.delta_ap_cv.iloc[::-1]])
axis.axvline(0, color='black', linewidth=.8)
axis.set(xlabel='AP CV(mask gốc) − AP CV(mask bỏ cột)', title='Ablation trên train-CV — hybrid 75%, seed 41')
fig.tight_layout()
fig.savefig(HINH / 'ablation_feature_hybrid_75.png', dpi=160)
plt.show()

## 4. Diễn giải và bước chưa thể làm
**Kết quả đã chạy:** Ở 75%, hybrid trung bình 15,6/38 cột, AP validation 0,8939 so với 0,8970 của đủ cột; không mask nào trong 30 mask vượt đủ cột cùng cutoff. Jaccard hybrid 75% là 0,2568, gần ngẫu nhiên cùng kích thước 0,2490. Hybrid thắng mask ngẫu nhiên tốt nhất trong tám lần chấm ở 3/5 seed 75% và 2/5 seed 50%; chưa đủ chứng minh ưu thế ổn định. Bỏ submission_rate khỏi hybrid seed 41 ở 75% làm AP train-CV giảm 0,004821; bỏ năm cột khác lại làm AP CV tăng rất nhẹ. Xem báo cáo kiểm chứng trong bao_cao để đọc toàn bộ bảng và giới hạn.

Phải đọc **cùng lúc** AP CV, AP validation, Recall, số cột, đối chứng ngẫu nhiên và Jaccard. Kết quả âm cũng là bằng chứng. Ablation chỉ đánh giá đóng góp có điều kiện ở một mask. Chưa có học kỳ mới hoàn toàn độc lập: các kỳ OULAD hiện có đã xuất hiện trong lịch sử nghiên cứu. Tách lại một kỳ từ cùng nguồn chỉ là kiểm tra độ nhạy, không biến nó thành test nguyên sơ. Muốn xác nhận thật, cần khóa trước evaluator/cutoff/alpha/budget/quy tắc chọn mask, lấy kỳ học mới chưa xem, rồi so đủ cột, HCO, VNLS, hybrid và mask ngẫu nhiên với Accuracy, Precision, Recall, F1, AP, số cột và thời gian; không điều chỉnh theo tập đó sau khi mở kết quả.